# 🔩 Fastener Company Finder — Google Colab

Tool tổng hợp **website + email** các công ty **sản xuất / nhập khẩu / phân phối** fasteners, screws, threaded rod, studs, washers... ở **Mỹ và Châu Âu**.

Notebook này chỉ là **adapter mỏng**: cell 1 tự tải bản core mới nhất (`fastener_finder.py`) từ repo GitHub [nghiant96/fastener-contact-tool](https://github.com/nghiant96/fastener-contact-tool) — nâng cấp ở core là notebook tự có, kể cả khi bạn đang dùng bản *Copy in Drive*.

**Cách dùng:** chạy các cell từ trên xuống. Cell 2 kết nối Google Drive để **mọi kết quả tự lưu vào Drive**, không mất khi Colab ngắt kết nối.

In [ ]:
#@title 1️⃣ Cài thư viện + tải core mới nhất từ GitHub
%pip -q install ddgs pandas openpyxl requests

import importlib, urllib.request
CORE_URL = ("https://raw.githubusercontent.com/nghiant96/"
            "fastener-contact-tool/main/fastener_finder.py")
urllib.request.urlretrieve(CORE_URL, "fastener_finder.py")
import fastener_finder as ff
importlib.reload(ff)
print("✅ Đã nạp core. Vùng đang bật:", list(ff.REGIONS))

In [ ]:
#@title 2️⃣ Kết nối Google Drive — MỌI kết quả sẽ tự lưu vào đây
SAVE_TO_DRIVE = True  #@param {type:"boolean"}

import os
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/fastener_data"
    os.makedirs(OUT_DIR, exist_ok=True)
else:
    OUT_DIR = "/content"   # ổ tạm — MẤT khi Colab ngắt kết nối!

MASTER = f"{OUT_DIR}/fastener_companies_master.csv"
print("📁 Dữ liệu lưu tại:", OUT_DIR)
print("   (mở Drive → thư mục 'fastener_data' là thấy file)")

In [ ]:
#@title 3️⃣ (Tuỳ chọn) Chỉnh cấu hình — bỏ qua nếu dùng mặc định
# Chỉ giữ vài nước:
# ff.REGIONS = {k: v for k, v in ff.REGIONS.items() if k in ["USA", "Germany"]}
# Thêm nước mới:
# ff.REGIONS["Czech"] = (["Czech Republic"], "cz-cs")
# Thêm sản phẩm / bang Mỹ:
# ff.PRODUCTS.append("anchor bolts")
# ff.US_STATES.append("Nevada")
print(f"Sản phẩm: {ff.PRODUCTS}\nVai trò: {ff.ROLES}\nVùng: {list(ff.REGIONS)}")

In [ ]:
#@title 4️⃣ Quét tìm công ty (kết quả tự lưu vào Drive)
DEEP_SCAN = False  #@param {type:"boolean"}
# DEEP_SCAN=True: quét theo TỪNG BANG Mỹ + từ khoá ASTM/SAE/UNC — rất nhiều
# truy vấn (hàng nghìn), chỉ nên bật khi có nhiều thời gian.

OUT_CSV = f"{OUT_DIR}/fastener_companies.csv"
df = ff.run(out_csv=OUT_CSV, deep_geo=DEEP_SCAN)
df.head(20)

## 📧 Tìm EMAIL trên các website đã thu thập

Vào từng website, đọc trang chủ + tối đa 3 trang liên hệ (contact / kontakt / impressum...), trích tối đa 5 email/công ty — bắt được cả email bị Cloudflare che và dạng `name (at) domain (dot) com`.

Kết quả tách rõ: `emails` (cùng domain — đáng tin), `emails_external` (email lạ), `email_status` (found / not_found / timeout / blocked / error), `email_found_on`. **Chạy lại cell là tự resume** — chỉ quét website chưa xong, checkpoint ghi vào Drive mỗi 50 website.

In [ ]:
#@title 5️⃣ Quét email (tự lưu Drive + resume được)
df2 = ff.add_emails_to_csv(OUT_CSV)
df2[["company_name", "website", "emails", "email_status"]].head(20)

In [ ]:
#@title 6️⃣ (Tuỳ chọn) Tải file về máy
from google.colab import files
for f in (OUT_CSV.replace(".csv", "_with_emails.csv"),
          OUT_CSV.replace(".csv", "_with_emails.xlsx")):
    if os.path.exists(f):
        files.download(f)
    else:
        print("chưa có:", f, "— chạy cell 5 trước")

## 🔄 (Tuỳ chọn) Chế độ CHẠY LIÊN TỤC

Quét lặp vô hạn: mỗi vòng ~60 truy vấn ngẫu nhiên (tự bốc cả **bang của Mỹ** và từ khoá đặc thù vùng), **chỉ thêm công ty MỚI** vào file tổng trên Drive. Chạy càng lâu danh sách càng dài; Colab ngắt kết nối cũng không mất dữ liệu — mở lại chạy tiếp là tích luỹ tiếp.

⚠️ Colab miễn phí tự ngắt sau ~90 phút không tương tác (tối đa 12h). Chạy 24/7 thật sự: dùng bản standalone trên máy — `python fastener_finder.py --loop 30`.

In [ ]:
#@title 7️⃣ Chạy liên tục (dừng bằng ⏹ Stop — Drive đã lưu sau mỗi vòng)
ff.run_forever(
    interval_minutes=15,   # nghỉ giữa các vòng
    queries_per_cycle=60,  # số truy vấn mỗi vòng
    master_csv=MASTER,     # file tổng trên Drive
)

In [ ]:
#@title 8️⃣ Quét email cho file tổng (chạy sau khi loop đã gom được nhiều)
df3 = ff.add_emails_to_csv(MASTER)
print(df3["email_status"].value_counts().to_string())

## 💡 Mẹo
- **Kết quả nằm ở đâu**: Google Drive → thư mục `fastener_data`. Mở bằng Google Sheets được luôn.
- **Cột chất lượng**: ưu tiên `qualification_status = qualified`, duyệt tay nhóm `review`; `confidence_score` càng cao càng tin; `verified_country` là nước suy từ đuôi tên miền.
- **Chấm điểm lại file cũ**: `ff.qualify_dataframe(df)`.
- **Bị chặn / ratelimit**: `ff.SLEEP_RANGE = (3, 6)` rồi chạy lại.
- Luôn kiểm tra tay trước khi gửi email hàng loạt.